# Q2 调用 API 辅助分析


### 预处理数据加载

In [1]:
# 导入必要的库
import pandas as pd
import numpy as np

# 读取数据集并进行预处理
df = pd.read_csv('/Users/halo/Desktop/数据分析及实践实验/实验5/subdata.csv', index_col=0)

# 处理目标变量缺失的行
df_clean = df.dropna(subset=['TEACHBEHA'])

# 处理剩余缺失值
numerical_cols = df_clean.select_dtypes(include=['number']).columns.tolist()
categorical_cols = df_clean.select_dtypes(exclude=['number']).columns.tolist()

# 数值型特征使用均值填充
for col in numerical_cols:
    if col != 'TEACHBEHA' and df_clean[col].isnull().sum() > 0:
        df_clean[col] = df_clean[col].fillna(df_clean[col].mean())

# 分类特征使用众数填充
for col in categorical_cols:
    if df_clean[col].isnull().sum() > 0:
        df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])

# 检查数据已完全预处理
print(f"预处理后数据形状: {df_clean.shape}")
print(f"预处理后缺失值总数: {df_clean.isnull().sum().sum()}")

# 显示数据框的前几行
display(df_clean.head())

# 显示目标变量的描述性统计
print("\nTEACHBEHA 的描述性统计:")
display(df_clean['TEACHBEHA'].describe())

预处理后数据形状: (1054, 25)
预处理后缺失值总数: 0


/var/folders/s5/x61j5mg13qjcd3vpv32rjn6h0000gn/T/ipykernel_88214/1468278797.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean[col] = df_clean[col].fillna(df_clean[col].mean())


,CNTSCHID,Region,STRATUM,LANGTEST,PRIVATESCH,SCHLTYPE,STRATIO,SCHSIZE,RATCMP1,RATCMP2,...,CLSIZE,CREACTIV,EDUSHORT,STAFFSHORT,STUBEHA,TEACHBEHA,SCMCEG,W_SCHGRNRABWT,W_FSTUWT_SCH_SUM,SENWT
0,72400001,72413,ESP9028,156.0,private,2.0,1.2000,75.0,0.836264,1.0,...,28.0,0.0,1.0283,1.0953,-0.6266,-0.1274,-0.1868,9.44357,796.01006,6.85830
1,72400002,72415,ESP1532,156.0,public,3.0,7.6379,443.0,0.714300,1.0,...,23.0,1.0,0.0686,0.4300,0.5078,1.1612,0.9042,1.56862,121.29469,1.13920
2,72400003,72409,ESP9018,156.0,private,1.0,10.4452,1572.0,0.260400,1.0,...,23.0,1.0,-0.9490,-0.5869,-2.0719,-1.4190,0.9042,16.53833,1538.38583,12.01080
3,72400004,72406,ESP0611,156.0,public,3.0,12.6935,1574.0,0.078900,1.0,...,23.0,2.0,0.4292,0.2633,-0.6744,0.2266,0.1571,1.54470,80.31823,1.12182
4,72400005,72417,ESP9038,156.0,public,3.0,9.5074,965.0,0.632900,1.0,...,28.0,1.0,2.1861,1.5598,0.2913,-0.9033,0.2870,9.15181,975.57348,6.64641



TEACHBEHA 的描述性统计:


count    1054.000000
mean       -0.099276
std         0.967054
min        -2.090400
25%        -0.599275
50%        -0.117700
75%         0.601500
max         3.787900
Name: TEACHBEHA, dtype: float64

## （a）无提示词输出

In [ ]:
from openai import OpenAI

# 初始化 OpenAI 客户端，使用 DeepSeek 的 API
client = OpenAI(
    api_key="sk-",  # 请替换为您的API密钥
    base_url="https://api.deepseek.com/v1"
)

# 设计输入提示词 - 直接问问题
prompt = """[Question] 已知给定pandas.DataFrame实例df，请编写Python代码求特征STUBEHA与TEACHBEHA、EDUSHORT与STAFFSHORT的相关系数。"""

# 发送请求
response = client.chat.completions.create(
    model="deepseek-chat",
    messages=[
        {"role": "user", "content": prompt}
    ],
    temperature=0.1,  # 低温度以获得更确定性的输出
    max_tokens=1000
)

# 输出结果
print("提示词:")
print(prompt)
print("\n大语言模型的输出:")
print(response.choices[0].message.content)

提示词:
[Question] 已知给定pandas.DataFrame实例df，请编写Python代码求特征STUBEHA与TEACHBA、EDUSHORT与STAFFSHORT的相关系数。

大语言模型的输出:
# 计算DataFrame中指定特征的相关系数

要计算DataFrame中特征STUBEHA与TEACHBA、EDUSHORT与STAFFSHORT的相关系数，可以使用pandas的`corr()`方法。以下是Python代码实现：

```python
import pandas as pd

# 假设df是已经存在的DataFrame
# 计算STUBEHA与TEACHBA的相关系数
corr1 = df['STUBEHA'].corr(df['TEACHBA'])

# 计算EDUSHORT与STAFFSHORT的相关系数
corr2 = df['EDUSHORT'].corr(df['STAFFSHORT'])

# 打印结果
print(f"STUBEHA与TEACHBA的相关系数: {corr1:.4f}")
print(f"EDUSHORT与STAFFSHORT的相关系数: {corr2:.4f}")
```

或者，你也可以一次性计算所有列的相关系数矩阵，然后提取需要的值：

```python
# 计算整个相关系数矩阵
corr_matrix = df[['STUBEHA', 'TEACHBA', 'EDUSHORT', 'STAFFSHORT']].corr()

# 提取特定相关系数
corr1 = corr_matrix.loc['STUBEHA', 'TEACHBA']
corr2 = corr_matrix.loc['EDUSHORT', 'STAFFSHORT']

# 打印结果
print(f"STUBEHA与TEACHBA的相关系数: {corr1:.4f}")
print(f"EDUSHORT与STAFFSHORT的相关系数: {corr2:.4f}")
```

注意：
1. 相关系数的范围在-1到1之间
2. 1表示完全正相关，-1表示完全负相关，0表示无线性相关
3. 默认计算的是皮尔逊相关系数
4. 如果数据中有缺失值，计算前需要先处理缺失值


In [5]:
import pandas as pd

# 假设df是已经存在的DataFrame
# 计算STUBEHA与TEACHBA的相关系数
corr1 = df['STUBEHA'].corr(df['TEACHBEHA'])

# 计算EDUSHORT与STAFFSHORT的相关系数
corr2 = df['EDUSHORT'].corr(df['STAFFSHORT'])

# 打印结果
print(f"STUBEHA与TEACHBEHA的相关系数: {corr1:.4f}")
print(f"EDUSHORT与STAFFSHORT的相关系数: {corr2:.4f}")

STUBEHA与TEACHBEHA的相关系数: 0.5663
EDUSHORT与STAFFSHORT的相关系数: 0.5237


## （b）带提示词输出

In [ ]:
from openai import OpenAI

# 初始化 OpenAI 客户端，使用 DeepSeek 的 API
client = OpenAI(
    api_key="sk-xxx",  # 请替换为您的API密钥
    base_url="https://api.deepseek.com/v1"
)

# 设计输入提示词 - 提供样例的提示词
prompt_with_examples = """下面是一些使用pandas计算数据统计信息的示例:

[Question] 已知给定pandas.DataFrame实例df，请编写一段Python代码输出列A和列B的平均值。
[Answer] ```print(df['A'].mean(), df['B'].mean())```

[Question] 已知给定pandas.DataFrame实例df，请编写一段Python代码计算列C的标准差。
[Answer] ```print(df['C'].std())```

[Question] 已知给定pandas.DataFrame实例df，请编写一段Python代码按照列D对数据进行排序。
[Answer] ```sorted_df = df.sort_values(by='D')
print(sorted_df)```

[Question] 已知给定pandas.DataFrame实例df，请编写Python代码求特征STUBEHA与TEACHBEHA、EDUSHORT与STAFFSHORT的相关系数。
[Answer]"""

# 发送请求
response = client.chat.completions.create(
    model="deepseek-chat",
    messages=[
        {"role": "user", "content": prompt_with_examples}
    ],
    temperature=0.1,  # 低温度以获得更确定性的输出
    max_tokens=1000
)

# 输出结果
print("提示词(含样例):")
print(prompt_with_examples)
print("\n大语言模型的输出:")
print(response.choices[0].message.content)

提示词(含样例):
下面是一些使用pandas计算数据统计信息的示例:

[Question] 已知给定pandas.DataFrame实例df，请编写一段Python代码输出列A和列B的平均值。
[Answer] ```print(df['A'].mean(), df['B'].mean())```

[Question] 已知给定pandas.DataFrame实例df，请编写一段Python代码计算列C的标准差。
[Answer] ```print(df['C'].std())```

[Question] 已知给定pandas.DataFrame实例df，请编写一段Python代码按照列D对数据进行排序。
[Answer] ```sorted_df = df.sort_values(by='D')
print(sorted_df)```

[Question] 已知给定pandas.DataFrame实例df，请编写Python代码求特征STUBEHA与TEACHBEHA、EDUSHORT与STAFFSHORT的相关系数。
[Answer]

大语言模型的输出:
```python
# 计算STUBEHA与TEACHBEHA的相关系数
corr1 = df['STUBEHA'].corr(df['TEACHBEHA'])
# 计算EDUSHORT与STAFFSHORT的相关系数
corr2 = df['EDUSHORT'].corr(df['STAFFSHORT'])

print(f"STUBEHA与TEACHBEHA的相关系数: {corr1}")
print(f"EDUSHORT与STAFFSHORT的相关系数: {corr2}")
```


In [7]:
# 计算STUBEHA与TEACHBEHA的相关系数
corr1 = df['STUBEHA'].corr(df['TEACHBEHA'])
# 计算EDUSHORT与STAFFSHORT的相关系数
corr2 = df['EDUSHORT'].corr(df['STAFFSHORT'])

print(f"STUBEHA与TEACHBEHA的相关系数: {corr1}")
print(f"EDUSHORT与STAFFSHORT的相关系数: {corr2}")

STUBEHA与TEACHBEHA的相关系数: 0.5663393467193838
EDUSHORT与STAFFSHORT的相关系数: 0.5237122998210914


## （c）比较分析两种提示方式的输出差异

通过观察上述两种不同提示方式的输出，可以发现：

1. **输出格式的一致性**：带有样例的提示词更容易让模型生成格式统一、风格一致的代码，符合示例中的格式要求。

2. **代码的简洁性**：样例会引导模型生成更简洁的代码，避免不必要的注释或解释。

3. **解决方案的准确性**：通过样例，模型能更好地理解任务需求，提供更准确的解决方案。

4. **命名约定的继承**：样例可以引导模型遵循特定的变量命名约定。

5. **输出可预测性**：提供样例使模型输出更加可预测，减少了随机性。

总体而言，少量但高质量的样例能显著提高API生成代码的质量和一致性，特别是在需要特定格式输出的场景中。